In [ ]:
# Cell 1: TEST - RML2018 evaluation (31-aligned data prep + pinned best checkpoint)
from pathlib import Path
import ast
import random

import h5py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import load_model

np.random.seed(42)
random.seed(42)

notebook_dir = Path().resolve()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir

cfg_path = project_root / 'configs' / 'local_data_paths.yaml'
if cfg_path.exists():
    cfg = yaml.safe_load(cfg_path.read_text()) or {}
    dcfg = cfg.get('datasets', {}).get('rml2018', {})
    h5_path = Path(dcfg.get('hdf5', '/scratch/rameyjm7/datasets/RML2018/GOLD_XYZ_OSC.0001_1024.hdf5'))
    classes_path = Path(dcfg.get('classes', '/scratch/rameyjm7/datasets/RML2018/classes.txt'))
else:
    h5_path = Path('/scratch/rameyjm7/datasets/RML2018/GOLD_XYZ_OSC.0001_1024.hdf5')
    classes_path = Path('/scratch/rameyjm7/datasets/RML2018/classes.txt')

best_ckpt_txt = project_root / 'models' / 'rml2018' / 'checkpoints' / 'best_checkpoint.txt'
default_model = project_root / 'models' / 'rml2018' / 'rml2018_lstm_rnn.keras'
if best_ckpt_txt.exists():
    candidate = Path(best_ckpt_txt.read_text().strip())
    model_path = candidate if candidate.exists() else default_model
else:
    model_path = default_model

print('RML2018 dataset:', h5_path)
print('RML2018 classes:', classes_path)
print('RML2018 model  :', model_path)

assert h5_path.exists(), f'Missing dataset: {h5_path}'
assert classes_path.exists(), f'Missing classes file: {classes_path}'
assert model_path.exists(), f'Missing model: {model_path}'


outputs_dir = project_root / 'outputs' / 'rml2018_eval'
outputs_dir.mkdir(parents=True, exist_ok=True)
print('Outputs dir   :', outputs_dir)



In [ ]:
# Cell 2: Build 31-aligned eval split (SNR > -6 dB, <= 30 dB, stratified random split)
SNR_MIN_DB = -6
SNR_MAX_DB = 30
MAX_SAMPLES_PER_CLASS = 3000
TEST_SPLIT = 0.20
RANDOM_STATE = 42


def parse_classes(path: Path):
    text = path.read_text()
    return [c.strip(" '") for c in ast.literal_eval(text.split('=')[-1].strip())]


def load_all_rml2018(h5_path, classes_txt, snr_min_db=-6, snr_max_db=30, max_per_class=3000):
    class_list = parse_classes(classes_txt)

    with h5py.File(h5_path, 'r') as f:
        X, Y, Z = f['X'][:], f['Y'][:], f['Z'][:]

    per_class = {cls: [] for cls in class_list}

    for i in range(len(X)):
        snr = int(Z[i][0])
        if (snr > snr_min_db) and (snr <= snr_max_db):
            cls = class_list[int(Y[i].argmax())]
            sig = np.hstack([X[i], np.full((1024, 1), snr, dtype=np.float32)])
            per_class[cls].append(sig)

    if max_per_class:
        for k in per_class:
            random.shuffle(per_class[k])
            per_class[k] = per_class[k][:max_per_class]

    X_out, y_out = [], []
    for cls, arr in per_class.items():
        X_out.extend(arr)
        y_out.extend([cls] * len(arr))

    return np.array(X_out, dtype=np.float32), np.array(y_out)


X_all, y_all = load_all_rml2018(
    h5_path,
    classes_path,
    snr_min_db=SNR_MIN_DB,
    snr_max_db=SNR_MAX_DB,
    max_per_class=MAX_SAMPLES_PER_CLASS,
)

le = LabelEncoder()
y_all_enc = le.fit_transform(y_all)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_all,
    y_all_enc,
    test_size=TEST_SPLIT,
    stratify=y_all_enc,
    random_state=RANDOM_STATE,
)

X_eval, y_eval = X_te, y_te

print('SNR filter: snr >', SNR_MIN_DB, 'and snr <=', SNR_MAX_DB)
print('Train shape:', X_tr.shape)
print('Eval shape :', X_eval.shape)
print('Num classes:', len(le.classes_))



In [ ]:
# Cell 3: Evaluate model and plot confusion matrix
model = load_model(model_path, compile=False)
y_pred = np.argmax(model.predict(X_eval, verbose=0), axis=1)

acc = accuracy_score(y_eval, y_pred)
report_txt = classification_report(y_eval, y_pred, target_names=le.classes_, zero_division=0)
print(f'RML2018 evaluation accuracy (31-aligned protocol): {acc:.4f}')
print(report_txt)

report_path = outputs_dir / '41_rml2018_classification_report.txt'
report_path.write_text(report_txt)
print('Saved report:', report_path)

cm = confusion_matrix(y_eval, y_pred, labels=np.arange(len(le.classes_)))
plt.figure(figsize=(12, 10))
sns.heatmap(cm, cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix - RML2018 (31-aligned protocol)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
cm_png = outputs_dir / '41_rml2018_confusion_matrix.png'
plt.savefig(cm_png, dpi=180)
print('Saved confusion matrix:', cm_png)
plt.show()




In [ ]:
# Cell 4: Plot and save accuracy per SNR per modulation
import csv

# Build a per-(class,snr) balanced evaluation slice from raw RML2018
MAX_PER_CLASS_PER_SNR = 200


def load_rml2018_per_snr(h5_path, classes_txt, snr_min_db=-6, snr_max_db=30, max_per_class_per_snr=200):
    classes_list = parse_classes(classes_txt)

    with h5py.File(h5_path, 'r') as f:
        X = f['X'][:]
        Y = f['Y'][:]
        Z = f['Z'][:]

    buckets = {}  # (class_name, snr) -> list of samples

    for i in range(len(X)):
        snr = int(Z[i][0])
        if (snr > snr_min_db) and (snr <= snr_max_db):
            cls_name = classes_list[int(Y[i].argmax())]
            key = (cls_name, snr)
            buckets.setdefault(key, [])
            if len(buckets[key]) < max_per_class_per_snr:
                sig = np.hstack([X[i], np.full((1024, 1), snr, dtype=np.float32)])
                buckets[key].append(sig.astype(np.float32))

    xs, ys, snrs = [], [], []
    for (cls_name, snr), arr in buckets.items():
        for s in arr:
            xs.append(s)
            ys.append(cls_name)
            snrs.append(snr)

    X_eval_snr = np.asarray(xs, dtype=np.float32)
    y_labels = np.asarray(ys)
    snr_vals = np.asarray(snrs, dtype=np.int64)
    return X_eval_snr, y_labels, snr_vals


X_snr_eval, y_snr_labels, snr_vals = load_rml2018_per_snr(
    h5_path,
    classes_path,
    snr_min_db=SNR_MIN_DB,
    snr_max_db=SNR_MAX_DB,
    max_per_class_per_snr=MAX_PER_CLASS_PER_SNR,
)

# Map labels into the same LabelEncoder order used for model eval in this notebook
y_snr_true = le.transform(y_snr_labels)
y_snr_pred = np.argmax(model.predict(X_snr_eval, verbose=0), axis=1)

snr_unique = np.array(sorted(np.unique(snr_vals)), dtype=np.int64)
class_names = list(le.classes_)
acc_grid = np.full((len(class_names), len(snr_unique)), np.nan, dtype=np.float32)

for c_idx in range(len(class_names)):
    cls_mask = y_snr_true == c_idx
    for j, snr in enumerate(snr_unique):
        m = cls_mask & (snr_vals == snr)
        if np.any(m):
            acc_grid[c_idx, j] = float(np.mean(y_snr_pred[m] == y_snr_true[m]))

# Plot heatmap
plt.figure(figsize=(16, 10))
sns.heatmap(
    acc_grid,
    cmap='viridis',
    vmin=0.0,
    vmax=1.0,
    xticklabels=snr_unique,
    yticklabels=class_names,
    cbar_kws={'label': 'Accuracy'},
)
plt.title('RML2018 Accuracy per SNR per Modulation')
plt.xlabel('SNR (dB)')
plt.ylabel('Modulation')
plt.tight_layout()

snr_mod_png = outputs_dir / '41_rml2018_accuracy_per_snr_per_modulation.png'
plt.savefig(snr_mod_png, dpi=180)
print('Saved per-SNR/per-modulation heatmap:', snr_mod_png)
plt.show()

# Save raw table to CSV
snr_mod_csv = outputs_dir / '41_rml2018_accuracy_per_snr_per_modulation.csv'
with open(snr_mod_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['modulation'] + [str(s) for s in snr_unique])
    for i, mod in enumerate(class_names):
        row = [mod] + [
            '' if np.isnan(acc_grid[i, j]) else f"{acc_grid[i, j]:.6f}"
            for j in range(len(snr_unique))
        ]
        writer.writerow(row)

print('Saved per-SNR/per-modulation table:', snr_mod_csv)
print('Per-SNR eval sample count:', X_snr_eval.shape[0])



In [ ]:
# Cell 5: Plot and save line charts (overall accuracy vs SNR, per-modulation vs SNR)
import csv

# Recompute predictions on the per-SNR eval slice built in Cell 4
y_snr_pred = np.argmax(model.predict(X_snr_eval, verbose=0), axis=1)

# Overall accuracy by SNR
overall_acc = []
for snr in snr_unique:
    idx = np.where(snr_vals == snr)[0]
    overall_acc.append(float(np.mean(y_snr_pred[idx] == y_snr_true[idx])) * 100.0)

# Per-modulation lines from acc_grid (already in [0,1])
per_mod_percent = acc_grid * 100.0

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

axes[0].plot(snr_unique, overall_acc, marker='o', color='blue')
axes[0].set_title('Recognition Accuracy vs. SNR (RML2018)')
axes[0].set_xlabel('SNR (dB)')
axes[0].set_ylabel('Accuracy (%)')
axes[0].grid(True, alpha=0.4)

for i, mod in enumerate(class_names):
    axes[1].plot(snr_unique, per_mod_percent[i], marker='o', linewidth=1.2, label=mod)
axes[1].set_title('Accuracy vs. SNR per Modulation Type (RML2018)')
axes[1].set_xlabel('SNR (dB)')
axes[1].set_ylabel('Accuracy (%)')
axes[1].grid(True, alpha=0.4)
axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)

plt.tight_layout()
line_png = outputs_dir / '41_rml2018_accuracy_vs_snr_line_plots.png'
plt.savefig(line_png, dpi=180)
print('Saved line charts:', line_png)
plt.show()

overall_csv = outputs_dir / '41_rml2018_accuracy_vs_snr_line.csv'
with open(overall_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['snr_db', 'accuracy_percent'])
    for s, a in zip(snr_unique, overall_acc):
        writer.writerow([int(s), f"{a:.6f}"])
print('Saved overall line data:', overall_csv)



In [ ]:
# Cell 6: Load and plot continuation training curves from outputs/rml2018 history
import json

history_dir = project_root / 'outputs' / 'rml2018'
history_files = sorted(history_dir.glob('*.history.json'))

if not history_files:
    print('No history files found in', history_dir)
else:
    # Use the newest history file by modification time
    latest = max(history_files, key=lambda p: p.stat().st_mtime)
    print('Using history file:', latest)

    hist = json.loads(latest.read_text())
    acc = hist.get('accuracy', [])
    val_acc = hist.get('val_accuracy', [])
    loss = hist.get('loss', [])
    val_loss = hist.get('val_loss', [])

    if not (acc and val_acc and loss and val_loss):
        print('History file missing one or more required series: accuracy, val_accuracy, loss, val_loss')
    else:
        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        plt.plot(acc, label='train_accuracy')
        plt.plot(val_acc, label='val_accuracy')
        plt.title(f'Continuation Accuracy ({len(acc)} epochs)')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.grid(True)
        plt.legend()

        plt.subplot(1, 2, 2)
        plt.plot(loss, label='train_loss')
        plt.plot(val_loss, label='val_loss')
        plt.title(f'Continuation Loss ({len(loss)} epochs)')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.grid(True)
        plt.legend()

        plt.tight_layout()
        curves_png = outputs_dir / '41_rml2018_continuation_training_curves.png'
        plt.savefig(curves_png, dpi=180)
        print('Saved training curves:', curves_png)
        plt.show()



: 